In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import numpy as np
import plotly.figure_factory as ff



In [4]:
import sys
!{sys.executable} -m pip install geopandas
import geopandas as gpd

  Using cached geopandas-1.1.3-py3-none-any.whl.metadata (2.3 kB)
  Using cached pyogrio-0.12.1-cp313-cp313-macosx_12_0_arm64.whl.metadata (5.9 kB)
  Using cached pyproj-3.7.2-cp313-cp313-macosx_14_0_arm64.whl.metadata (31 kB)
Using cached geopandas-1.1.3-py3-none-any.whl (342 kB)
Using cached pyogrio-0.12.1-cp313-cp313-macosx_12_0_arm64.whl (23.7 MB)
Using cached pyproj-3.7.2-cp313-cp313-macosx_14_0_arm64.whl (4.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [geopandas]/3 [geopandas]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [13]:
fp = "tmp0kekil41.csv"
df = pd.read_csv(fp)


/var/folders/pc/7dj9yy6d0yj9lhpgvnlq86wr0000gp/T/ipykernel_61269/516374522.py:2: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fp)


In [14]:
#print(df.head())
print(df.iloc[1])
violation_counts = df["violdesc"].value_counts()
#print(violation_counts)
violation_levels = df["viol_level"].value_counts()
#print(violation_levels)


businessname                                   1000 Degrees Pizza
dbaname                                                       NaN
legalowner                                           KHOSLA VIPAN
namelast                                          Pasquriello LLC
namefirst                                    Kenneth Pasquariello
licenseno                                                  313440
issdttm                                    2017-08-14 12:49:37+00
expdttm                                    2020-01-01 04:59:00+00
licstatus                                                Inactive
licensecat                                                     FS
descript                                        Eating & Drinking
result                                                    HE_Fail
resultdttm                                 2018-03-20 14:54:25+00
violation                                         22-4-601/602.11
viol_level                                                     **
violdesc  

In [15]:
df = df[df["licensecat"] == "FS"]


df = df.drop_duplicates()
df["violdttm"] = pd.to_datetime(df["violdttm"], errors = "coerce")
df["issdttm"] = pd.to_datetime(df["issdttm"], errors = "coerce")
df["expdttm"] = pd.to_datetime(df["expdttm"], errors = "coerce")

df["violdesc"] = df["violdesc"].str.strip()
df["businessname"] = df["businessname"].str.strip()


#Zip code cleaning
df = df.dropna(subset=['zip'])
df['zip'] = df['zip'].astype(str)
df['zip'] = df['zip'].str.split('.').str[0]
df = df[df['zip'].str.isnumeric()]
df['zip'] = df['zip'].str.zfill(5)
print(df['zip'].unique())
print("Number of unique ZIP codes:", df['zip'].nunique())


['02108' '02118' '02110' '02131' '02125' '02116' '02114' '02135' '02129'
 '02111' '02134' '02109' '02128' '02132' '02115' '02122' '02215' '02199'
 '02210' '02130' '02127' '02119' '02113' '02136' '02188' '02120' '02124'
 '02126' '02201' '02121' '02163' '02446' '02467']
Number of unique ZIP codes: 33


In [16]:
df_fail = df[df["viol_status"] == "Fail"]
main_levels = ["*", "**", "***"]
df_fail = df_fail[df_fail["viol_level"].isin(main_levels)]

df_fail['year'] = df_fail['violdttm'].dt.year
print(df_fail['zip'].unique())
viol_business_count = df_fail.groupby(['businessname','year','viol_level']).size().reset_index(name='num_violations')
print(viol_business_count.head(10))
viol_zip_count = df_fail.groupby(['zip','year','viol_level']).size().reset_index(name='num_violations')
print(viol_zip_count.head(10))


['02108' '02118' '02131' '02125' '02116' '02114' '02135' '02129' '02111'
 '02110' '02134' '02109' '02128' '02132' '02115' '02122' '02215' '02199'
 '02210' '02130' '02127' '02119' '02113' '02136' '02188' '02120' '02124'
 '02126' '02201' '02121' '02163' '02446' '02467']
               businessname  year viol_level  num_violations
0  100 Percent Delicia Food  2013          *              16
1  100 Percent Delicia Food  2013        ***               1
2  100 Percent Delicia Food  2014          *               3
3  100 Percent Delicia Food  2015          *              13
4  100 Percent Delicia Food  2015        ***               1
5  100 Percent Delicia Food  2016          *              18
6  100 Percent Delicia Food  2016        ***               5
7  100 Percent Delicia Food  2017          *              19
8  100 Percent Delicia Food  2017        ***               3
9  100 Percent Delicia Food  2018          *              11
     zip  year viol_level  num_violations
0  02108  2007    

In [28]:
#Prepare data for location
df_fail = df_fail[df_fail['location'].notnull()]

def extract_latlon(x):
    try: 
        if isinstance(x, (tuple, list)):
            return float(x[0]), float(x[1])
        elif isinstance(x, str):
            parts = x.replace('(', '').replace(')', '').split(',')
            return float(parts[0]), float(parts[1])
    except:
        return np.nan, np.nan

df_fail['lat'], df_fail['lon'] = zip(*df_fail['location'].apply(extract_latlon))


df_fail = df_fail[df_fail['lat'].notnull() & df_fail['lon'].notnull()]

# Ensure types are numeric
df_fail['lat'] = df_fail['lat'].astype(float)
df_fail['lon'] = df_fail['lon'].astype(float)





In [30]:
# Round coordinates to ~0.01 degrees (~1 km) to define clusters
df_fail['lat_bin'] = df_fail['lat'].round(2)   # replace 'latitude' with your column name
df_fail['lon_bin'] = df_fail['lon'].round(2)  # replace 'longitude' with your column name

# Combine rounded coordinates to define a cluster
df_fail['cluster'] = df_fail['lat_bin'].astype(str) + "_" + df_fail['lon_bin'].astype(str)

print(df_fail[['businessname', 'cluster']].drop_duplicates().head(10))


                                        businessname       cluster
0                                 1000 Degrees Pizza  42.36_-71.06
13                              1000 Washington Cafe  42.35_-71.06
48                          100 Percent Delicia Food  42.28_-71.12
316                                        110 Grill  42.33_-71.06
533                              11 Dining -16th Fl.  42.35_-71.07
564                        125 Nashua St. Cafe (MGH)  42.37_-71.06
744   150 Boylston St. Dining Room @ Emerson College  42.35_-71.07
877                          163 Vietnamese Sandwich  42.35_-71.06
1120                                1928 Beacon Hill  42.36_-71.07
1168                                1928 Rowes Wharf  42.36_-71.05


In [58]:
viol_cluster_count = (
    df_fail.groupby(['cluster', 'year', 'viol_level'])
    .size()
    .reset_index(name = 'num_violations')
)

print(viol_cluster_count)



           cluster  year viol_level  num_violations
0     42.24_-71.13  2008          *              10
1     42.24_-71.13  2008         **               1
2     42.24_-71.13  2008        ***               7
3     42.24_-71.13  2009          *              18
4     42.24_-71.13  2009         **               2
...            ...   ...        ...             ...
5794    42.4_-71.0  2019         **               4
5795    42.4_-71.0  2019        ***               1
5796    42.4_-71.0  2022         **               2
5797    42.4_-71.0  2022        ***               1
5798    42.4_-71.0  2024          *               2

[5799 rows x 4 columns]


In [40]:
# Compute cluster coordinates and total violations
cluster_summary = (
    df_fail.groupby(['cluster', 'viol_level'])
    .agg(
        num_violations=('businessname', 'count'),  # count failed restaurants
        lat_avg=('lat', 'mean'),             # replace 'latitude' with your column name
        lon_avg=('lon', 'mean')             # replace 'longitude' with your column name
    )
    .reset_index()
)

# Optionally, compute total violations per cluster for sizing
cluster_total = cluster_summary.groupby('cluster').agg(
    total_violations=('num_violations', 'sum')
).reset_index()

cluster_summary = cluster_summary.merge(cluster_total, on='cluster')

In [60]:
severity_map = {"*": 1, "**": 2, "***": 3}
df_fail['severity_score'] = df_fail['viol_level'].map(severity_map)

In [70]:
# Round lat/lon to create small grid cells (~100m)
df_fail['lat_bin'] = df_fail['lat'].round(3)  # 0.001 deg ~ 100m
df_fail['lon_bin'] = df_fail['lon'].round(3)

# Aggregate violations per grid cell and severity
grid_summary = (
    df_fail.groupby(['lat_bin', 'lon_bin', 'viol_level'])
    .agg(total_severity = ('severity_score', 'sum'),
         businesses =("businessname", lambda x: ", ".join(sorted(set(x)))),
         num_violations = ("businessname", 'count')
    )
    .reset_index()
)
print(grid_summary)
# For a general heat, sum over all violation levels
grid_summary_total = (
    df_fail.groupby(['lat_bin', 'lon_bin'])
    .size()
    .reset_index(name='num_violations')
)

      lat_bin  lon_bin viol_level  total_severity  \
0      42.238  -71.132          *              62   
1      42.238  -71.132         **              10   
2      42.238  -71.132        ***              48   
3      42.242  -71.127          *               4   
4      42.242  -71.127         **              14   
...       ...      ...        ...             ...   
3238   42.394  -71.000         **              42   
3239   42.394  -71.000        ***              27   
3240   42.395  -71.000          *               6   
3241   42.395  -71.000         **              12   
3242   42.395  -71.000        ***              15   

                                             businesses  num_violations  
0                                 Readville Tavern Inc.              62  
1                                 Readville Tavern Inc.               5  
2                                 Readville Tavern Inc.              16  
3                                   Stop & Shop No. 413            

In [71]:

cluster_summary['viol_level'] = cluster_summary['viol_level'].astype(str)
fig = px.density_mapbox(
    grid_summary,                   
    lat='lat_bin',
    lon='lon_bin',
    z='total_severity',                   
    radius=15,
    hover_name = 'businesses',
    hover_data = {
        'total_severity': True,
        'num_violations': True,
        'lat_bin': False,
        'lon_bin': False
    },                             
    center=dict(lat=df_fail['lat'].mean(), lon=df_fail['lon'].mean()),  
    zoom=12,
    mapbox_style="carto-positron",
    title="Geospatial Heatmap of Failed Restaurant Violations"
)

fig.show()

/var/folders/pc/7dj9yy6d0yj9lhpgvnlq86wr0000gp/T/ipykernel_61269/1569978487.py:2: DeprecationWarning:

*density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [ ]:
fig = px.line(
    viol_zip_count, 
    x = 'year', 
    y = 'num_violations', 
    color = 'viol_level',
    line_group = 'zip',
    hover_name = 'zip',
    markers = True,
    title = 'Timeline of Violations by ZIP Code (2006-2026)'
)
trace = (
    viol_zip_count[['zip', 'viol_level']]
    .drop_duplicates()
    .sort_values(['viol_level', 'zip'])
)

zips = sorted(viol_zip_count['zip'].unique())
zip_buttons = []
zip_buttons.append(dict(
    label = "All ZIPs",
    method = 'update',
    args =[
        {'visible': [True] * len(fig.data)}, 
        {'title': "Violations Over Time for All ZIPs"}
    ]


))

for z in zips:
    zip_buttons.append(dict(
        label = str(z),
        method = "update",
        args = [
            {'visible': [row.zip == str(z) for row in trace.itertuples()]},
            {"title": f"Violations Over Time for ZIP {z}"}
        ]
    ))

fig.update_layout(
    updatemenus = [
        dict(
            buttons = [
                dict(
                    label = "All ZIPs",
                    method = 'update',
                    args = [{"visible": [True]*len(fig.data)},
                            {"title": "Violations Over Time for All ZIPs"}]
                )
            ] + zip_buttons,
            direction = 'down',
            showactive = True
        )
    ]
)

fig.update_layout(
    xaxis = dict(dtick = 1),
    yaxis_title = "Number of Violations",
    legend_title = "Violation Level",
    hovermode = 'x unified',
    title_x = 0.5,
    legend = dict(
        title = 'Violation Level',
        orientation = 'v',
        x = 1.02,
        y = 1
    )
)
fig.show()

In [46]:
fig.write_html('YehS.graphHomework7.html')